# Aula 11 · Trapézio e Simpson

Esta aula apresenta o [capítulo 11 do site](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/). A ideia central: **a integral é a área sob a curva, e dá para calculá-la cortando a área em faixas** — trapézios (uma reta por faixa) ou parábolas (Simpson), com a precisão que se quiser.

**Ao fim da aula você consegue:**

1. deduzir as regras do trapézio e de Simpson 1/3 e aplicá-las à mão;
2. programar as duas regras e conferir a ordem do erro;
3. reconhecer que Simpson exige um número par de faixas;
4. integrar uma tabela de medições, inclusive com espaçamento irregular.

**Roteiro:** 🧩 · 1. a área · 2. 🧑‍🏫 trapézio · 3. 🧑‍🏫 Simpson · 4. a ordem · 5. quando Simpson erra · 6. confira · 7. outra área · 🎯 prática · 🧩 o genérico · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Indústria farmacêutica — o genérico é equivalente?**
>
> *Para um genérico ser aprovado, a ANVISA exige um estudo de **bioequivalência**:
> voluntários tomam o remédio de referência e o genérico, e a concentração no sangue
> é medida em várias horas. Duas contas decidem: a **área sob a curva** (AUC, a
> exposição total do corpo ao remédio) e o **pico** (Cmax). As duas razões
> genérico/referência têm de ficar entre **80 % e 125 %**. "**O nosso genérico
> passa?**"*

As coletas de sangue não são igualmente espaçadas. No fim da aula, você calcula as
duas áreas a partir das medições.

## 1. A área sob a curva

A probabilidade de uma medição cair a menos de um desvio-padrão da média é a área
sob a curva normal entre $-1$ e $1$. Essa integral não tem fórmula.

📖 [capítulo 11 · A área sob a curva](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#a-area-sob-a-curva)

In [ ]:
# 📦 dados prontos — só rode esta célula
# A curva normal (estatística): a área entre -1 e 1 é a chance de uma medição
# cair a menos de um desvio-padrão da média.
def normal(z):
    return np.exp(-z**2 / 2) / np.sqrt(2 * np.pi)

**✍️ Passo 1.** Desenhe `normal` de −3 a 3 e calcule a área de um único trapézio entre −1 e 1: `(1 - (-1)) / 2 * (normal(-1) + normal(1))`.

In [ ]:
# ✍️ passo 1

**Preveja:** um trapézio só vai dar mais ou menos que a área de verdade (0,6827)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Menos**: 0,4839. A reta entre as pontas corta fora o "morro" da curva no meio.
Com mais faixas, cada reta acompanha melhor a curva.

📖 [capítulo 11 · A área sob a curva](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#a-area-sob-a-curva)

</details>

## 2. No quadro: a regra do trapézio

📖 [capítulo 11 · No quadro: a regra do trapézio](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#no-quadro-a-regra-do-trapezio)

### 🧑‍🏫 No quadro — a regra do trapézio

Caderno de papel aberto. No quadro:

1. $n$ faixas de largura $h = (b-a)/n$;
2. em cada faixa, a área de um trapézio;
3. somando: cada ponto do meio aparece em duas faixas;
4. o erro proporcional a $h^2$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ I \approx \frac{h}{2}\Big(y_0 + 2\,(y_1 + \cdots + y_{n-1}) + y_n\Big) $$

</details>

**✍️ Passo 2.** Calcule o trapézio com `n = 4` na normal: `h`, os pontos `x = np.linspace(-1, 1, 5)`, `y = normal(x)` e a fórmula (com um laço para somar os pontos do meio).

In [ ]:
# ✍️ passo 2

**Preveja:** com 4 faixas, quantas casas batem com 0,6827?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`0.6725`: só a primeira casa. Com 16 faixas, sairia 0,6821 — três casas.

</details>

### 🎯 Sua vez — A regra do trapézio

Escreva `trapezio(f, a, b, n)`. Use `np.sum(y[1:-1])` para somar os pontos do meio.

In [ ]:
def trapezio(f, a, b, n):
    # sua solução aqui
    pass

In [ ]:
confere(trapezio, [
    ((normal, -1, 1, 16), 0.6820590314814196),
    ((np.exp, 0, 1, 1), 1.8591409142295225),
])

<details>
<summary><b>💡 Dica</b></summary>

`y[1:-1]` é tudo menos as pontas. As pontas pesam 1, o meio pesa 2.

</details>

## 3. No quadro: a regra de Simpson

📖 [capítulo 11 · No quadro: a regra de Simpson](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#no-quadro-a-regra-de-simpson)

### 🧑‍🏫 No quadro — a regra de Simpson 1/3

Caderno de papel aberto. No quadro:

1. uma parábola por três pontos vizinhos (o polinômio do capítulo 9);
2. a integral dela: $\frac{h}{3}(y_0 + 4y_1 + y_2)$;
3. emendando parábolas de duas em duas faixas: pesos 1, 4, 2, 4, ..., 4, 1;
4. por que $n$ tem de ser par; o erro proporcional a $h^4$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ I \approx \frac{h}{3}\Big(y_0 + 4\,(y_1 + y_3 + \cdots) + 2\,(y_2 + y_4 + \cdots) + y_n\Big) $$

</details>

> 🧰 **Python: fatia com passo**
>
> `y[inicio:fim:passo]` pega de `passo` em `passo`. `y[1:-1:2]` são as posições 1, 3,
> 5, ... (sem a última); `y[2:-1:2]`, as posições 2, 4, ... (sem a última).

In [ ]:
# 🧰 exemplo — só rode e veja a saída
y = [10, 11, 12, 13, 14, 15, 16]
print(y[1:-1:2], y[2:-1:2])

### 🎯 Sua vez — A regra de Simpson

Escreva `simpson(f, a, b, n)` (suponha `n` par).

In [ ]:
def simpson(f, a, b, n):
    # sua solução aqui
    pass

In [ ]:
confere(simpson, [
    ((normal, -1, 1, 8), 0.6827109757132983),
    ((np.exp, 0, 1, 2), 1.7188611518765928),
])

<details>
<summary><b>💡 Dica</b></summary>

`np.sum(y[1:-1:2])` pesa 4, `np.sum(y[2:-1:2])` pesa 2, as pontas pesam 1, tudo vezes `h / 3`.

</details>

**✍️ Passo 3.** Compare `trapezio` e `simpson` na normal, com `n = 8`.

In [ ]:
# ✍️ passo 3

**Preveja:** qual chega mais perto de 0,682689?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Simpson: 0,682711 contra 0,680164 do trapézio — com os **mesmos** 9 valores da
função. A parábola acompanha a curvatura que a reta ignora.

📖 [capítulo 11 · No quadro: a regra de Simpson](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#no-quadro-a-regra-de-simpson)

</details>

## 4. A ordem do erro

📖 [capítulo 11 · A ordem do erro](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#a-ordem-do-erro)

**✍️ Passo 4.** Para $e^x$ em $[0, 1]$ (exato: $e - 1$), calcule o erro das duas regras com `n = 8` e `n = 16`, e a razão erro(8)/erro(16) de cada uma.

In [ ]:
# ✍️ passo 4

**Preveja:** que razões o quadro prevê?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Trapézio: **4** ($2^2$). Simpson: **16** ($2^4$). Como no capítulo 2, a ordem
aparece na razão.

📖 [capítulo 11 · A ordem do erro](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#a-ordem-do-erro)

</details>

## 5. Quando Simpson dá errado

📖 [capítulo 11 · Quando Simpson dá errado](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#quando-simpson-da-errado)

**✍️ Passo 5.** Calcule `simpson(np.exp, 0, 1, n)` para `n` = 4, 5, 6 e 7, e o erro de cada um.

In [ ]:
# ✍️ passo 5

**Preveja:** o que acontece com `n` ímpar?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Com 5 e 7, erros de 16 % e 12 %, contra $10^{-5}$ com 4 e 6. Com $n$ ímpar, as
parábolas não se emendam e os pesos saem errados — sem aviso nenhum.

📖 [capítulo 11 · Quando Simpson dá errado](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#quando-simpson-da-errado)

</details>

> ⚠️ **Armadilha.** $n$ é o número de **faixas**; o de **pontos** é $n + 1$. Uma tabela com 7
medições tem 6 faixas (par): Simpson serve. Com 8 medições, 7 faixas: não serve.

## 6. Confira com a biblioteca

📖 [capítulo 11 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#confira-com-a-biblioteca)

> 🧰 **Comando novo: `from scipy.integrate import quad`**
>
> `quad(f, a, b)` calcula a integral com um método adaptativo e devolve **dois**
> números: o valor e uma estimativa do erro. Por isso: `valor, erro = quad(f, a, b)`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
from scipy.integrate import quad

valor, erro = quad(np.exp, 0, 1)
print(valor, erro)

**✍️ Passo 6.** Confira a área da normal entre −1 e 1 com o `quad`.

In [ ]:
# ✍️ passo 6

**Preveja:** o seu Simpson com `n = 16` acertou quantas casas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`quad` dá 0,6826894921. O Simpson com 16 faixas (0,682691) acerta 5 casas.

📖 [capítulo 11 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#confira-com-a-biblioteca)

</details>

## 7. Mesmo método, outra área

**Topografia.** A largura de um terreno foi medida a cada 10 m ao longo de uma linha
de base: 0; 12,4; 18,9; 22,1; 25,6; 24,0; 19,8; 15,2; 0 m.

📖 [capítulo 11 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Calcule a área do terreno pelo trapézio e por Simpson (com `h = 10`), usando o array das larguras direto na fórmula — sem função nenhuma.

In [ ]:
# ✍️ passo 7

**Preveja:** os dois resultados vão ser parecidos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

1380,0 m² pelo trapézio e 1411,3 m² por Simpson: 2 % de diferença. Com pontos
tão espaçados, a curvatura do contorno pesa, e Simpson é o mais confiável.

📖 [capítulo 11 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/11-trapezio-simpson/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: integrar uma tabela com **espaçamento irregular**, em que nem o trapézio de faixas iguais nem Simpson servem direto.

## 🧩 Resolvendo o problema

> *"**O nosso genérico passa?**"* — a farmacêutica responsável.

A célula 📦 tem as médias das concentrações medidas nos voluntários. Repare nas horas
de coleta: de meia em meia hora no começo, espaçadas no fim.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Estudo de bioequivalência: concentração no sangue (mg/L) de voluntários, nas
# horas de coleta, com o remédio de referência e com o genérico.
horas = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 12]
referencia = [0.00, 8.65, 11.91, 12.71, 12.41, 10.75, 8.94, 6.02, 4.04, 1.81]
generico = [0.00, 5.98, 8.84, 9.95, 10.11, 9.17, 7.76, 5.22, 3.45, 1.49]

plt.figure()
plt.plot(horas, referencia, "o-", label="referência")
plt.plot(horas, generico, "o-", label="genérico")
plt.xlabel("horas depois da dose")
plt.ylabel("concentração (mg/L)")
plt.grid()
plt.legend()
plt.show()

### 🎯 Sua vez — A área de uma tabela qualquer

Escreva `trapezio_dados(x, y)`, que soma a área de cada trapézio,
$\dfrac{(x_{i+1} - x_i)(y_i + y_{i+1})}{2}$, faixa por faixa — cada uma com a
sua largura.

In [ ]:
def trapezio_dados(x, y):
    # sua solução aqui
    pass

In [ ]:
confere(trapezio_dados, [
    (([0, 1, 3], [2, 4, 0]), 7.0),
    (([0, 0.5, 2, 4], [1, 1, 1, 1]), 4.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Um acumulador e um laço de `0` a `len(x) - 2`.

</details>

As duas contas da ANVISA:

In [ ]:
auc_ref = trapezio_dados(horas, referencia)
auc_gen = trapezio_dados(horas, generico)
if auc_ref is not None:
    print("AUC referência:", auc_ref, "  AUC genérico:", auc_gen)
    print("razão das AUC: ", auc_gen / auc_ref)
    print("razão dos picos (Cmax):", max(generico) / max(referencia))

<details>
<summary><b>▶ O que os números dizem</b></summary>

A razão das áreas é **0.829**: dentro da faixa de 0,80 a 1,25. A exposição
total ao remédio é equivalente. Mas a razão dos picos é **0.795**:
**abaixo** de 0,80. O genérico é absorvido mais devagar e não chega à mesma
concentração máxima — o que pode importar num antibiótico, que precisa de um pico
alto para matar as bactérias.

**O genérico não passa**, e o motivo não aparece na integral, e sim no pico. O
laboratório precisa rever a formulação (a velocidade de dissolução do comprimido).
A regra do trapézio sobre as coletas é exatamente como a AUC é calculada nos estudos
de verdade.

</details>

## 📋 A lista

Abra a [Lista 11](https://lacouth.github.io/metodos_telecom-site/listas/lista11/). O **Exercício 01** é à mão (✏️): trapézio e Simpson com
duas faixas. Comece por ele, no papel.

**a)** Qual o valor exato de $\int_1^3 \frac{1}{x+1}\,dx$?

<details>
<summary><b>▶ Resposta</b></summary>

A primitiva é $\ln(x+1)$: $\ln 4 - \ln 2 = \ln 2 = 0{,}693147$ — e não $\ln 3$.

</details>

Termine o exercício e siga para o **Exercício 02**, o trapézio como função.

## 🚪 Antes de sair

**1.** Por que os pontos do meio pesam 2 no trapézio?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque cada um é o lado direito de uma faixa e o lado esquerdo da seguinte: entra em dois trapézios.

</details>

**2.** Com o mesmo número de pontos, por que Simpson erra tanto menos?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Porque troca a curva por parábolas, que acompanham a curvatura; o erro cai com $h^4$, e não com $h^2$.

</details>

**3.** Por que a AUC do genérico foi calculada pelo trapézio, e não por Simpson?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque as coletas não são igualmente espaçadas: Simpson, na forma do capítulo, exige faixas de mesma largura (e em número par). O trapézio faixa por faixa aceita qualquer espaçamento.

</details>

## 🏠 Para casa

- Refaça no papel a dedução dos pesos 1, 4, 1 de Simpson, a partir da parábola.
- Termine a [Lista 11](https://lacouth.github.io/metodos_telecom-site/listas/lista11/).
- Leia o começo do [capítulo 12](https://lacouth.github.io/metodos_telecom-site/unidade6-integrais/12-integrando-dados/): e se a
  gente quiser a integral **acumulada**, instante por instante?